# Tutorial 2 — FOL Formulas: Connectives & Quantifiers

Logical connectives (¬, ∧, ∨, →, ↔) and quantifiers (∀, ∃) are evaluated as
**fuzzy operations on truth-value tensors**.

The fuzzy operators live in a `Logic` bundle attached to an `Interpretation` (`interp.logic`).
Formula strings are parsed once (outside any loop) and re-evaluated on different data.

| Pattern | String syntax |
|---|---|
| Conjunction | `P(x) & Q(x)` |
| Disjunction | `P(x) ∨ Q(x)` |
| Negation | `not P(x)` or `¬P(x)` |
| Implication | `P(x) -> Q(x)` or `P(x) → Q(x)` |
| Universal | `forall x: P(x)` or `∀ x: P(x)` |
| Existential | `exists x: P(x)` or `∃ x: P(x)` |
| Diagonal (joint) | `∀ (x, y): Sim(x, y)` |
| Guarded | `∀ x \| Mask(x): P(x)` |

> **Quantifiers are non-greedy.** `∀ x: P(x) & Q(x)` parses as `(∀ x: P(x)) & Q(x)` — the
> quantifier binds only to the immediately following unary/atomic expression. Wrap the intended scope in parentheses: `∀ x: (P(x) & Q(x))`. This mirrors
> how quantifier scope works in standard mathematical notation; see the callout in section 1 below
> for a worked example.

In [ ]:
import torch
import torch.nn as nn
import sltn
from sltn import Signature, Real, Interpretation, Logic

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Setup

A small signature with two Gaussian-centred predicates and a similarity predicate.

In [ ]:
# Declare the signature and its elements
sig = Signature("t2")
sig.sort("S")
sig.variable("x", "S")
sig.variable("y", "S")
sig.predicate("P", ["S"])
sig.predicate("Q", ["S"])
sig.predicate("Sim", ["S", "S"])
print(sig)


# Declare the interpretation of the signature
interp = Interpretation(sig).to(device)

# Interpret sort on a type
R2 = sltn.Type("R2", shape=(2,)) # 2-D real vectors
interp["S"] = R2

# Interpret the signature
mu_p = torch.tensor([0.0, 0.0], device=device)
mu_q = torch.tensor([1.0, 1.0], device=device)
interp["P"]   = lambda v: torch.exp(-torch.norm(v - mu_p, dim=-1, keepdim=True))
interp["Q"]   = lambda v: torch.exp(-torch.norm(v - mu_q, dim=-1, keepdim=True))
interp["Sim"] = lambda u, v: torch.exp(-torch.norm(u - v, dim=-1, keepdim=True))

# Interpret the variables with random tensors
interp["x"] = torch.randn(8, 2, device=device)
interp["y"] = torch.randn(6, 2, device=device)

print(interp)

## 1 — Connectives

Connective operators follow the **product** configuration by default.  
`sig.parse` resolves symbol names to `Symbol` objects; calling the interpretation on the parsed
formula, `interp(phi)`, applies the fuzzy operators.

In [ ]:
formulas = {
    "P(x)":        sig.parse("P(x)"),
    "not P(x)":    sig.parse("not P(x)"),
    "P(x) & Q(x)": sig.parse("P(x) & Q(x)"),
    "P(x) ∨ Q(x)": sig.parse("P(x) ∨ Q(x)"),
    "P(x) → Q(x)":  sig.parse("P(x) → Q(x)"),
}

for name, phi in formulas.items():
    r = interp(phi)
    print(f"{name:<20s}  shape={list(r.value.shape)}  mean={r.value.mean().item():.3f}")


## 2 — Quantifiers

`∀ x: φ(x)` aggregates over the variable axis with **pMeanError** (strict, default).  
`∃ x: φ(x)` uses **pMean** (permissive, default).  
The variable axis is aggregated — the result is a scalar `sltn.Tensor`.

In [ ]:
phi_all = sig.parse("forall x: P(x)")
phi_ex  = sig.parse("exists x: P(x)")

# interp["x"] and interp["y"] are set — no assignment dict needed
r_all = interp(phi_all)
r_ex  = interp(phi_ex)

print("∀x P(x):", round(r_all.value.item(), 4), "  shape:", list(r_all.value.shape))
print("∃x P(x):", round(r_ex.value.item(),  4), "  shape:", list(r_ex.value.shape))

# Nested: ∀x ∃y Sim(x, y)
phi_nested = sig.parse("forall x: exists y: Sim(x, y)")
r_nested   = interp(phi_nested)
print("∀x∃y Sim(x, y) :", round(r_nested.value.item(), 4))


A quantifier binds only to the smallest following expression.
`∀ x: P(x) & Q(x)` therefore means `(∀ x: P(x)) & Q(x)`, **not** `∀ x: (P(x) & Q(x))`. If you want
the quantifier to scope over the whole conjunction, add explicit parentheses.

In [ ]:
phi_no_parens = sig.parse("forall x: P(x) & Q(x)")   # == (forall x: P(x)) & Q(x)
phi_scoped    = sig.parse("forall x: (P(x) & Q(x))")  # quantifier scopes over the conjunction

r_no_parens = interp(phi_no_parens)
r_scoped    = interp(phi_scoped)

print("forall x: P(x) & Q(x)   ->", "shape:", list(r_no_parens.value.shape))   # not a scalar!
print("forall x: (P(x) & Q(x)) ->", "shape:", list(r_scoped.value.shape))      # scalar

## 3 — Diagonal (joint) quantification

`∀ (x, y): φ(x, y)` treats `x` and `y` as **jointly sampled**: the $i$-th `x`‑individual
is paired with the $i$-th `y`-individual (no Cartesian product).
Both variables must have the same batch size.
Pass plain tensors in the assignment dict — the framework wraps them with the correct axis layout.
Use this for paired-constraint axioms such as "each positive/negative pair satisfies …".


In [ ]:
# Diagonal quantification: pass paired tensors of equal batch size in the assignment dict.
# The framework wraps them automatically — no explicit sltn.Tensor construction needed.
x8 = torch.randn(8, 2, device=device)
y8 = torch.randn(8, 2, device=device)

phi_diag  = sig.parse("∀ (x, y): Sim(x, y)")    # paired
phi_indep = sig.parse("∀ x: ∀ y: Sim(x, y)")    # Cartesian then aggregate

r_diag  = interp(phi_diag,  {"x": x8, "y": y8})
r_indep = interp(phi_indep, {"x": x8, "y": y8})
print("∀(x,y) Sim  [diagonal] :", round(r_diag.value.item(),  4))
print("∀x∀y   Sim  [Cartesian]:", round(r_indep.value.item(), 4))


## 4 — Guarded quantification

`∀ x | G(x): φ(x)` aggregates only over individuals where `G(x) > 0.5`.
Masked-out individuals receive the neutral value (1 for ∀, 0 for ∃),
so unlabelled samples do not penalise the satisfaction score.
Useful for **partially labelled** datasets.

In [ ]:
data = interp["x"]
Mask = sig.predicate("Mask", ["S"])
labels = (data[:, 0] > 0).float().unsqueeze(1)   # 1 where x₀ > 0
interp[Mask] = lambda _: labels                     # precomputed; ignore tensor input

phi_guarded = sig.parse("∀ x | Mask(x): P(x)")
r_guarded   = interp(phi_guarded)
print("∀x|Mask P(x):", round(r_guarded.value.item(), 4))
print(f"  {labels.sum().int().item()} of {len(interp['x'])} individuals contribute")


## 5 — Updating the Logic

`interp.logic` is a mutable `Logic` bundle. Operators are read and written dictionary-style,
using the standard logic symbols (`&`, `∧`, `and`, …) as keys. Assigning a fresh `Logic()`
resets every operator to its default.

```python
interp.logic = Logic()                       # reset to the defaults
interp.logic["∀"] = AggregPMean(p=4)         # patch a single operator
interp.logic["&"] = conjunction.AndMin()
interp.logic["∀"]                            # reads back the current universal-quantifier operator
```

The default operators (set on construction) are shown in `print(interp)` under the *Logic* section.


In [ ]:
from sltn.interpretation.operators import aggregator, conjunction
from sltn import Logic

print("Default logic:")
print(interp.logic)

# create a copy of the interpretation with a different logic
interp2 = interp.with_updates()
interp2.logic["∀"] = aggregator.AggregPMean(p=2)
print("\nNew interpretation with different logic:")
print(interp2.logic)
print("\nOriginal interpretation's logic is unchanged:")
print(interp.logic)


# Patch a single operator in-place, using the standard logic symbol as key
interp.logic["&"] = conjunction.AndMin()
print("\nAfter setting interp.logic['&'] = AndMin():")
print("interp.logic['&'] :", interp.logic["&"])
print("interp.logic['∀']  :", interp.logic["∀"])

---

**Next:** [Tutorial 3 — Fuzzy Operators & Gradients](3-FuzzyOperators&Gradients.ipynb) looks at *why* the choice of connective and aggregation operator matters for learning.